In [3]:
!pip install sklearn-crfsuite
import nltk
import sklearn_crfsuite

#1. DOWNLOAD DATA
nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)

corpus = nltk.corpus.treebank.tagged_sents(tagset='universal')

#2. FEATURES
def get_features(sent, i):
    word = sent[i]
    features = {
        'word':     word.lower(),
        'suffix3':  word[-3:],
        'suffix2':  word[-2:],
        'is_title': word.istitle(),
        'is_upper': word.isupper(),
        'is_digit': word.isdigit(),
    }
    if i > 0:
        features['prev_word'] = sent[i-1].lower()  # ← key fix
    else:
        features['BOS'] = True
    return features

#3. PREPARE DATA
X = [[get_features([w for w,t in sent], i) for i in range(len(sent))]
     for sent in corpus[:3500]]

y = [[tag for word, tag in sent] for sent in corpus[:3500]]

#4. TRAIN
crf = sklearn_crfsuite.CRF(max_iterations=100)
crf.fit(X, y)
print("✅ Model Trained!")

#5. PREDICT
tokens     = "Nature is the source of life".split()
features   = [get_features(tokens, i) for i in range(len(tokens))]
prediction = crf.predict_single(features)

print("\n--- POS Tagging Result ---")
for word, tag in zip(tokens, prediction):
    print(f"  {word:15} → {tag}")